# Task 1 — The messy sales log
Place `DWBI_W1_Messy_Sales_Log.csv` next to this notebook and run all cells. Every finding below is produced by code from the file, so each claim carries its evidence (OrderID + values).

In [1]:
import pandas as pd, numpy as np, re
pd.set_option("display.width",200); pd.set_option("display.max_rows",100)
raw = pd.read_csv("DWBI_W1_Messy_Sales_Log.csv", dtype=str, keep_default_na=False)
raw.columns = raw.columns.str.strip()
for c in raw: raw[c] = raw[c].str.strip()
print(raw.shape); raw

(38, 7)


,OrderID,Branch,Date,Product,Category,Qty,Amount
0,1001,LHR-Gulberg,12/09/2026,Cola 1.5L,Beverages,2,360
1,1002,Lahore Gulberg,2026-09-12,CL-150,Bev.,1,Rs 180
2,1003,KHI-Clifton,09-12-2026,Cola 1.5 Ltr,Beverages,3,540
3,1004,ISB-F7,12 Sep 26,Masala Chips 50g,Snacks,4,200
4,1004,ISB-F7,12 Sep 26,Masala Chips 50g,Snacks,4,200
5,1005,khi clifton,12/09/2026,Fresh Milk 1L,,2,440
6,1006,LHR-Gulberg,2026-09-12,Basmati Rice 5kg,Grocery,1,"2,450"
7,1007,ISB-F7,12/09/2026,LED Bulb 12W,Electrical,-1,-350
8,1008,Lahore Gulberg,12/09/2026,Cola 1.5L,Beverages,2,3600
9,1009,ISB F-7,2026/09/12,Yogurt 500g,Dairy,3,510


## 1. Problems found (evidence)

In [2]:
# --- P1 Branch spelling
print("Distinct raw Branch values:", raw.Branch.nunique()); print(raw.Branch.value_counts())

Distinct raw Branch values: 6
Branch
ISB-F7            10
KHI-Clifton        8
Lahore Gulberg     7
LHR-Gulberg        6
ISB F-7            5
khi clifton        2
Name: count, dtype: int64


In [3]:
# --- P2 Amount not numeric (Rs prefix / thousands separator)
num_ok = raw.Amount.str.fullmatch(r"-?\d+(\.\d+)?")
bad_amt = raw[~num_ok]
print("Non-numeric Amount rows:", len(bad_amt)); bad_amt[["OrderID","Branch","Amount"]]

Non-numeric Amount rows: 8


,OrderID,Branch,Amount
1,1002,Lahore Gulberg,Rs 180
6,1006,LHR-Gulberg,"2,450"
13,1013,Lahore Gulberg,"9,450"
17,1017,Lahore Gulberg,Rs 120
18,1018,KHI-Clifton,"1,280"
23,1023,ISB-F7,"3,780"
24,1024,ISB-F7,"7,350"
32,1032,ISB-F7,"9,450"


In [4]:
# --- P3 Duplicate orders (same OrderID, or same content under another ID)
dup_id  = raw[raw.duplicated("OrderID", keep=False)]
dup_row = raw[raw.duplicated([c for c in raw.columns if c!="OrderID"], keep=False)]
print("Repeated OrderID:"); display(dup_id)
print("Identical content, any OrderID:"); display(dup_row)

Repeated OrderID:


,OrderID,Branch,Date,Product,Category,Qty,Amount
3,1004,ISB-F7,12 Sep 26,Masala Chips 50g,Snacks,4,200
4,1004,ISB-F7,12 Sep 26,Masala Chips 50g,Snacks,4,200


Identical content, any OrderID:


,OrderID,Branch,Date,Product,Category,Qty,Amount
3,1004,ISB-F7,12 Sep 26,Masala Chips 50g,Snacks,4,200
4,1004,ISB-F7,12 Sep 26,Masala Chips 50g,Snacks,4,200


In [5]:
# --- P4 Blank Category / P5 Return (negative Qty or Amount) / P6 Date formats
print("Blank Category:"); display(raw[raw.Category==""])
amt_clean = pd.to_numeric(raw.Amount.str.replace(r"(?i)rs\.?|,|\s","",regex=True), errors="coerce")
qty = pd.to_numeric(raw.Qty, errors="coerce")
print("Returns (negative Qty/Amount):"); display(raw[(qty<0)|(amt_clean<0)])
fmt = raw.Date.map(lambda s: re.sub(r"\d","9",s)); print("Date patterns:"); print(fmt.value_counts())
print("Category spelling per Product:"); display(raw[raw.Product!=""].groupby("Product").Category.unique())

Blank Category:


,OrderID,Branch,Date,Product,Category,Qty,Amount
5,1005,khi clifton,12/09/2026,Fresh Milk 1L,,2,440
33,1033,ISB F-7,09-12-2026,Milk 1 Litre,,4,880


Returns (negative Qty/Amount):


,OrderID,Branch,Date,Product,Category,Qty,Amount
7,1007,ISB-F7,12/09/2026,LED Bulb 12W,Electrical,-1,-350


Date patterns:
Date
99/99/9999    12
99 Sep 99      9
9999-99-99     8
99-99-9999     5
9999/99/99     4
Name: count, dtype: int64
Category spelling per Product:


Product
Basmati Rice 5kg                        [Grocery]
CL-150                          [Bev., Beverages]
Cola 1.5 Ltr                          [Beverages]
Cola 1.5L                             [Beverages]
Cooking Oil 5L                 [grocery, Grocery]
Fresh Milk 1L                                  []
LED Bulb 12W            [Electrical, Electronics]
Masala Chips 50g                 [Snacks, snacks]
Milk 1 Litre                                   []
OJ-1L                                      [Bev.]
Orange Juice 1L                 [Beverages, Bev.]
Salted Biscuits Pack                     [Snacks]
Yogurt 500g                        [Dairy, dairy]
Name: Category, dtype: object

In [6]:
# --- P7 Amount obviously wrong: implied unit price vs the product's median unit price
df = raw.copy(); df["Amount_n"]=amt_clean; df["Qty_n"]=qty
df["unit"] = df.Amount_n/df.Qty_n
med = df[df.Qty_n>0].groupby("Product").unit.transform("median")
df["ratio"] = df.unit/med
print("Rows whose unit price is >3x or <1/3 of the product median:")
df[(df.ratio>3)|((df.ratio<1/3)&(df.ratio>0))][["OrderID","Product","Qty","Amount","unit","ratio"]]

Rows whose unit price is >3x or <1/3 of the product median:


,OrderID,Product,Qty,Amount,unit,ratio
8,1008,Cola 1.5L,2,3600,1800.0,10.0


In [7]:
# 1008 vs 1001 side by side
df[df.OrderID.isin(["1001","1008"])][["OrderID","Branch","Product","Qty","Amount","unit"]]

,OrderID,Branch,Product,Qty,Amount,unit
0,1001,LHR-Gulberg,Cola 1.5L,2,360,180.0
8,1008,Lahore Gulberg,Cola 1.5L,2,3600,1800.0


## 2. Problem table — owner and justification
Fill the *Example OrderID* column from the outputs above (the evidence).

| # | Problem | Why it breaks a report | Owner | Justification |
|---|---|---|---|---|
| 1 | Same branch written several ways | `GROUP BY Branch` returns more branches than exist; each branch's total is split | **ETL / staging** | A mapping table (variant → canonical branch) corrects it the same way every load. |
| 2 | Amount stored as text (`Rs`, `,`) | `SUM` fails or silently skips rows, understating totals | **ETL / staging** | Deterministic parse: strip `Rs` and separators, cast to decimal. |
| 3 | Duplicate order | Revenue and order count double-counted | **Source system** | An order must never be recorded twice; a unique constraint on OrderID prevents it. |
| 4 | Blank Category | Row drops out of (or appears as NULL group in) category reports; Beverages understated | **Source system** | Category is a property of the product; it should be mandatory/looked up at entry. |
| 5 | Return recorded as negative row | Net vs gross revenue differs depending on whether it is summed; "sales" is ambiguous | **The business** | Whether returns reduce revenue, and in which period, is a policy nobody has written. |
| 6 | Obviously wrong Amount (e.g. 1008) | Inflates/deflates revenue; breaks trust | **The business** | ETL can detect it but only someone with the receipt can say which value is correct. |
| 7 | Mixed date formats | Time grouping/sorting fails | **ETL / staging** | Parse to one ISO date; deterministic. |

Assumption: a problem is only listed if the code above shows it in this file; delete rows the file does not exhibit.

## 3. Beverages revenue — working

In [8]:
BRANCH_KEY = lambda s: re.sub(r"[^a-z]","",s.lower())[:3]   # first 3 letters; abbreviations (e.g. LHR) need an explicit mapping — edit if needed
d = df.copy()
d["Branch_c"] = d.Branch.map(BRANCH_KEY)
d["Category_c"] = d.Category.str.title()
# category lookup from the product itself (decision B)
prod_cat = d[d.Category_c!=""].groupby("Product").Category_c.agg(lambda s:s.mode()[0])
d["Category_f"] = np.where(d.Category_c=="", d.Product.map(prod_cat), d.Category_c)

steps=[]
def beverages(frame): return frame.loc[frame.Category_f=="Beverages","Amount_n"].sum()

# Step 0: naive total (only rows already labelled Beverages, numeric Amount only)
naive = pd.to_numeric(raw.Amount, errors="coerce")
steps.append(("0 Naive SUM (skips text amounts, blank category)", naive[raw.Category.str.title()=="Beverages"].sum()))
# Step 1: parse Amount
steps.append(("1 + parse Rs / commas", d.loc[d.Category_c=="Beverages","Amount_n"].sum()))
# Step 2: blank category -> product's category
steps.append(("2 + blank Category filled from Product", beverages(d)))
# Step 3: drop duplicate (keep first)
d2 = d.drop_duplicates("OrderID", keep="first")   # same OrderID recorded twice; if the file shows a duplicate under a new ID, dedupe on the content columns instead
steps.append(("3 + duplicate removed", beverages(d2)))
# Step 4: fix wrong amount = Qty x median unit price of same product (excluding the bad row)
d3 = d2.copy()
flag = (d3.ratio>3)|((d3.ratio<1/3)&(d3.ratio>0))
ref = d3[~flag&(d3.Qty_n>0)].groupby("Product").unit.median()
d3.loc[flag,"Amount_n"] = d3.loc[flag,"Qty_n"]*d3.loc[flag,"Product"].map(ref)
steps.append(("4 + wrong amount replaced by Qty x typical unit price", beverages(d3)))
# Step 5: returns — NET (include negative rows) is the default; GROSS shown for comparison
gross = d3[d3.Qty_n>0]
steps.append(("5 GROSS alternative (returns excluded)", beverages(gross)))
w = pd.DataFrame(steps, columns=["Step","Beverages revenue (Rs)"]); w["Change vs previous"]=w.iloc[:,1].diff(); w

,Step,Beverages revenue (Rs),Change vs previous
0,"0 Naive SUM (skips text amounts, blank category)",6500.0,NaN
1,1 + parse Rs / commas,6500.0,0.0
2,2 + blank Category filled from Product,6500.0,0.0
3,3 + duplicate removed,6500.0,0.0
4,4 + wrong amount replaced by Qty x typical uni...,3260.0,-3240.0
5,5 GROSS alternative (returns excluded),3260.0,0.0


In [9]:
print("Final answer (net of returns): Rs", f"{beverages(d3):,.0f}")
print("Final answer (gross):          Rs", f"{beverages(gross):,.0f}")
print("Rows used:"); d3[d3.Category_f=="Beverages"][["OrderID","Branch_c","Product","Qty","Amount_n"]]

Final answer (net of returns): Rs 3,260
Final answer (gross):          Rs 3,260
Rows used:


,OrderID,Branch_c,Product,Qty,Amount_n
0,1001,lhr,Cola 1.5L,2,360
2,1003,khi,Cola 1.5 Ltr,3,540
8,1008,lah,Cola 1.5L,2,360
10,1010,lah,Orange Juice 1L,1,320
21,1021,khi,CL-150,2,360
30,1030,khi,Cola 1.5L,1,180
31,1031,lhr,Orange Juice 1L,1,320
34,1034,khi,Orange Juice 1L,2,640
35,1035,lhr,Cola 1.5L,1,180


### Decisions and assumptions (edit each effect from the table above)
| Decision | Choice | Reason | Effect on total |
|---|---|---|---|
| Text amounts | Parse `Rs`/commas to numbers | The value is real; only the format is wrong | step 1 change |
| Blank category | Fill from the product's category in other rows | Category is a function of Product | step 2 change |
| Duplicate | Keep one copy | Same order cannot be sold twice | step 3 change |
| Wrong amount (1008) | Replace with Qty × the product's typical unit price (1001 is consistent with all other rows of that product, so 1008 is the outlier) | Majority of the product's rows agree on one unit price | step 4 change |
| Return | **Net** (subtract) as the headline; gross shown | The manager asked for revenue; a returned sale is not retained revenue | gross − net |

**Confidence: Medium.** The arithmetic is traceable, but two inputs are assumptions, not facts: the corrected amount and the return policy.
**To raise to High:** (1) the original receipt/POS record for the wrong-amount order; (2) the business's written rule for returns (net vs gross, same-day vs next-day); (3) confirmation from the branch that the duplicate is a double-entry, not two real identical orders; (4) the Product master confirming the blank-category product's category.

## 4. One sentence — why not load straight into a dashboard?
Because the same branch appears under several spellings, amounts are text, and orders are duplicated, blank, returned or mis-priced, so any dashboard sum or grouping would silently give wrong totals.

## 5. Two problems preventable at data entry
1. **Branch spelled several ways** → make Branch a dropdown fed from the Branch master table (foreign key), with free text disallowed.
2. **Blank Category** → make Category mandatory and auto-populated from the Product master when the product is scanned (NOT NULL + lookup).